In [ ]:
pip install -r requirements.txt

In [ ]:
import os
import json
import random
from datetime import datetime
import tempfile
import base64
from pathlib import Path

import glob
import json

import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/src/functionapp")
#On windows use  sys.path.append(module_path+"\\src\\functionapp")

from ai_ocr.azure.openai_ops import load_image, get_size_of_base64_images
from ai_ocr.azure.images import convert_pdf_into_image
from ai_ocr.model import Config
from ai_ocr.chains import get_structured_data, get_client, perform_gpt_evaluation_and_enrichment
from ai_ocr.azure.doc_intelligence import get_ocr_results

from langchain_core.output_parsers.json import parse_json_markdown

from dotenv import load_dotenv
load_dotenv()

### Run the Solution once on the demo to produce an output.json

In [ ]:
input_directory = '../demo/medical-dataset/'
#input_directory = '../demo/default-dataset/'
#input_directory = '../demo/eval-dataset/'

system_prompt =  ''
with open(input_directory+'system_prompt.txt', 'r') as file_sys_prompt:
    system_prompt = file_sys_prompt.read()

output_schema = ''
with open(input_directory+'output_schema.json', 'r') as file_output_schema:
    output_schema = file_output_schema.read()

# Create a dict with content key to store the OCR results
ocr_result = {
    "content": ""
}

# Loop over directory and process all PDFs
for file in os.listdir(input_directory):
    if file.endswith(".pdf"):
        ocr_result["content"] += get_ocr_results(input_directory+file)
        print(f"OCR results:{ocr_result['content'] }")

        # Extract images from the PDF
        convert_pdf_into_image(input_directory+file)
    
 # Ensure the /tmp/ directory exists
imgs_path = os.path.join(os.getcwd(), "/tmp/")
os.makedirs(imgs_path, exist_ok=True)
    
# Determine the path for the temporary images
imgs = glob.glob(f"{imgs_path}/page*.png")
    
# Limit images by config
config = Config()
print(f"Config img size: {config.max_images}")
imgs = imgs[:config.max_images]
imgs = [load_image(img) for img in imgs]
print(f"Images count: {len(imgs)}")

# Check and reduce images total size if over 20MB
#max_size = 20 * 1024 * 1024  # 20MB
#while get_size_of_base64_images(imgs) > max_size:
#    imgs.pop()
    
# Get structured data
structured = get_structured_data(ocr_result["content"], system_prompt, output_schema, imgs)
    
# Parse structured data and return as JSON
x = parse_json_markdown(structured.content)  
response = json.dumps(x)

print(f'Response: {response}')

actual_output_path = "tmp/output.json"
with open(actual_output_path, 'w') as f:
    f.write(response)
    
print(actual_output_path);

## 1. Evaluating with the LLM

### Load the input (an output from previous LLM run) and evaluate using LLM as a judge

In [ ]:
input_directory = '../demo/medical-dataset/'
tmp_directory = 'tmp/'

output_schema = ''
with open(input_directory+'output_schema.json', 'r') as file_output_schema:
    output_schema = file_output_schema.read()
    
output_extraction = ''
with open(tmp_directory+'output.json', 'r') as file_output_extraction:
    output_extraction = file_output_extraction.read()
    
imgs_path = os.path.join(os.getcwd(), "/tmp/")
# Determine the path for the temporary images
imgs = glob.glob(f"{imgs_path}/page*.png")
    
# Limit images by config
config = Config()
print(f"Config img size: {config.max_images}")
imgs = imgs[:config.max_images]
imgs = [load_image(img) for img in imgs]
print(f"Images count: {len(imgs)}")
    

evaluation = perform_gpt_evaluation_and_enrichment(imgs, output_extraction, output_schema)

print(f'Response: {evaluation}')

evaluation_str = json.dumps(evaluation, indent=2)

actual_output_path = "tmp/evaluation.json"
with open(actual_output_path, 'w') as f:
    f.write(evaluation_str)
    
print(actual_output_path)


In [ ]:
# Delete all generated images created after processing
for file in os.listdir(imgs_path):
    if file.endswith(".jpeg") or file.endswith(".png"):
        image_path = os.path.join(imgs_path, file)
        try:
            os.remove(os.path.join(imgs_path, file))
            print(f"Deleted image: {image_path}")
        except Exception as e:
            print(f"Error deleting image {image_path}: {e}")

## 2. Evaluating with ground truth

### Load the input (an output from previous LLM run), ground truth and create a jsonl file

In [ ]:
import sys
if module_path not in sys.path:
    sys.path.append(module_path)

import json
import time
from pprint import pprint

def compile_jsonl(ground_truth_path, actual_output_path, output_file):
    # Read the ground truth JSON file
    with open(ground_truth_path, 'r') as gt_file:
        ground_truth = json.load(gt_file)

    with open(eval_schema_path, 'r') as eval_schema_file:
        eval_schema = json.load(eval_schema_file)


    # Open the output file
    with open(output_file, 'w') as out_file:
        # Iterate over each actual output JSON file
        with open(actual_output_path, 'r') as af:
            actual_data = json.load(af)
            # Combine ground truth and actual data into one object
            combined_data = {"ground_truth": ground_truth, "actual": actual_data, "eval_schema":eval_schema}
            # Write the combined data as a single line in the jsonl file
            out_file.write(json.dumps(combined_data) + '\n')


ground_truth_path = f"{module_path}/demo/default-dataset/ground_truth.json"
eval_data_path = f"{module_path}/demo/default-dataset/eval_data.jsonl"
eval_schema_path = f"{module_path}/demo/default-dataset/evaluation_schema.json"

compile_jsonl(ground_truth_path, actual_output_path, eval_data_path)



### Evaluate using ground truth

In [ ]:
from promptflow.evals.evaluate import evaluate
from src.evaluators.json_evaluator import JsonEvaluator
eval_data_path = f"{module_path}/demo/default-dataset/eval_data.jsonl"
with open(eval_data_path) as file:
    data = json.load(file)
    ground_truth = data["ground_truth"]
    evaluation_schema = data["eval_schema"]

evaluators = {}
evaluator_config = {}
default_match_evaluator_config = {}
json_evaluator = JsonEvaluator()
evaluators["json_evaluator"] = json_evaluator
evaluator_config["json_evaluator"] = {
    "actual": "${data.actual}",
    "ground_truth": "${data.ground_truth}",
    "eval_schema": "${data.eval_schema}"
}

timestamp = time.strftime("%m_%d.%H.%M.%S")
output_path = f"{module_path}/notebooks/outputs/output_{timestamp}.json"

results = evaluate(
    evaluation_name="test_eval_1",
    data=eval_data_path,
    evaluators=evaluators,
    evaluator_config=evaluator_config,
    output_path=output_path
)
pprint(results)


### Orginize all results in output files in a dataframe and print as a table

In [ ]:
import pandas as pd
import json

output_data_path = f"{module_path}/notebooks/outputs/"
dfs = []
merged_df = pd.DataFrame()
run_number = 1

filenames = sorted([f for f in os.listdir(output_data_path) if f.endswith(".json")])

# Loop through all files in the directory
for filename in filenames:
    if filename.endswith(".json"):  # Check if the file is a JSON file
        file_path = os.path.join(output_data_path, filename)
        
        # Load the JSON data
        with open(file_path, 'r') as file:
            data = json.load(file)
        
        # Convert the 'metrics' dictionary to a DataFrame
        df = pd.DataFrame.from_dict(data['metrics'], orient='index', columns=[filename])

        df = df[df.index.str.startswith('json_evaluator.CustomStringEvaluator')]
        df.reset_index(inplace=True)
        df.columns = ['Fields', f'Run {run_number}']
        
                # Merge DataFrames
        if merged_df.empty:
            merged_df = df
        else:
            merged_df = pd.merge(merged_df, df, on="Fields", how='outer')
        run_number += 1

merged_df['Fields'] = merged_df['Fields'].str.replace('json_evaluator.CustomStringEvaluator.', '')
merged_df['Average'] = merged_df.iloc[:, 1:].mean(axis=1)
merged_df = merged_df.round(1)

print(merged_df)

### Visualize the dataframe with seaborn library

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")

f, ax = plt.subplots(figsize=(4, 10))

merged_df.set_index('Fields', inplace=True)

sns.barplot(x="Average", y="Fields", data=merged_df, label="Fields", color="b")

plt.figure(figsize=(4, 10))
cmap = sns.color_palette(["blue", "orange"])
sns.heatmap(merged_df.iloc[:, :-1], annot=True, cmap=cmap, cbar=False, linewidths=.5, fmt='g', annot_kws={"size": 10, "color": "black"})
plt.title('Field Performance Across Runs')
plt.show()